# 03. Label Studio annotation을 MMYOLO 형식으로 export

이 노트북은 Label Studio project에서 원하는 annotation 또는 prediction을 골라 export합니다.

기본값은 `RUN_EXPORT=False`입니다. 먼저 설정값을 확인한 뒤, 실제 export할 때만 `RUN_EXPORT=True`로 바꾸세요.

자주 쓰는 조건:

- `SOURCE_TYPE = "ann"`: 사람이 직접 작업한 annotation을 export
- `SOURCE_TYPE = "pred"`: 모델이 만든 prediction을 export
- `ANN_USER_ID`: 특정 작업자 annotation만 골라내기 위한 user id
- `ANN_MIN_LEAD`: Label Studio의 lead_time 기준 필터. 보통 0 이상이면 작업한 annotation만 골라내는 용도로 사용합니다.
- `EXPORT_TYPE = "ann"`: annotation JSON만 저장
- `EXPORT_TYPE = "both"`: 이미지와 annotation을 같이 저장
- `INCLUDE_EMPTY_IMAGES = False`: 선택한 user/source의 bbox가 없는 task는 image에서도 제외

주의: export는 output 폴더에 파일을 생성합니다. dataset/export 결과는 Git에 넣지 마세요.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다.")

REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from labelstudio_bbox_tools.config import settings_from_env
from labelstudio_bbox_tools.exporters.project_export import export_project

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

PROJECT_ID = 244
OUT_DIR = Path("/share_ssd/ltb/Users/ltb/label_studio/운영서버_도메인맞춤_추가학습용/images/260623_steam_서초현장_260423/태깅완료")
# OUT_DIR = Path("/share_ssd/ltb/Users/ltb/label_studio/운영서버_도메인맞춤_추가학습용/images/260622_con8_개선/183_20220628_90156_ch01/태깅완료")

EXPORT_TYPE = "ann"      # "img", "ann", "both"
ANN_FORMAT = "mmyolo"    # "mmyolo", "yolo", "yolo_obb"
SOURCE_TYPE = "ann"      # "ann" 또는 "pred"

# SOURCE_TYPE="ann"일 때 필요합니다.
ANN_USER_ID = 1
ANN_MIN_LEAD = 0.0
ACCEPT_LEAD_TIME_NONE = True

# SOURCE_TYPE="pred"일 때 필요합니다.
PRED_MODEL_VER = None

# MMYOLO JSON의 images[].file_name을 어떤 방식으로 저장할지 선택합니다.
# absolute: 현재 서버 절대경로. 4090 내부 확인에는 편하지만 다른 서버 재사용성은 낮습니다.
# doc-root-relative: LABEL_STUDIO_DOC_ROOT 기준 상대경로. 다른 서버로 옮길 때 더 유리합니다.
MMYOLO_FILE_NAME = "absolute"

# False가 기본 추천값입니다.
# False: ANN_USER_ID/PRED_MODEL_VER 조건에 맞는 bbox가 없는 task는 images[]에서도 제외합니다.
# True: bbox가 없는 image도 MMYOLO images[]에 포함합니다. negative sample을 의도적으로 남길 때만 사용하세요.
INCLUDE_EMPTY_IMAGES = False

# True로 바꿔야 실제 export가 실행됩니다.
RUN_EXPORT = True

In [ ]:
print("PROJECT_ID:", PROJECT_ID)
print("OUT_DIR:", OUT_DIR)
print("EXPORT_TYPE:", EXPORT_TYPE)
print("ANN_FORMAT:", ANN_FORMAT)
print("SOURCE_TYPE:", SOURCE_TYPE)
print("ANN_USER_ID:", ANN_USER_ID)
print("ANN_MIN_LEAD:", ANN_MIN_LEAD)
print("INCLUDE_EMPTY_IMAGES:", INCLUDE_EMPTY_IMAGES)
print("RUN_EXPORT:", RUN_EXPORT)

In [ ]:
if not RUN_EXPORT:
    print("RUN_EXPORT=False 이므로 export를 실행하지 않습니다. 설정을 확인한 뒤 True로 바꾸세요.")
else:
    settings = settings_from_env(REPO_ROOT / ".env")
    result = export_project(
        project_id=PROJECT_ID,
        ls_url=settings.url,
        api_key=settings.api_key,
        out_dir=OUT_DIR,
        doc_root=settings.doc_root,
        export_type=EXPORT_TYPE,
        ann_format=ANN_FORMAT,
        source_type=SOURCE_TYPE,
        ann_user_id=ANN_USER_ID,
        ann_min_lead=ANN_MIN_LEAD,
        accept_lead_time_none=ACCEPT_LEAD_TIME_NONE,
        pred_model_ver=PRED_MODEL_VER,
        mmyolo_file_name=MMYOLO_FILE_NAME,
        include_empty_images=INCLUDE_EMPTY_IMAGES,
    )
    print(result)

In [ ]:
# Export 후 간단 확인용 cell입니다.
# annotations_mmyolo.json이 생성된 뒤 실행하세요.
import json
from collections import Counter

json_path = OUT_DIR / "annotations_mmyolo.json"
if json_path.is_file():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    cat_by_id = {c["id"]: c["name"] for c in data.get("categories", [])}
    counts = Counter(cat_by_id.get(a.get("category_id"), "<unknown>") for a in data.get("annotations", []))
    print("images:", len(data.get("images", [])))
    print("annotations:", len(data.get("annotations", [])))
    print("categories:", len(data.get("categories", [])))
    print(counts.most_common(20))
else:
    print("아직 annotations_mmyolo.json이 없습니다.")